# Earthquakes Gold

Executable SQL transformations and orchestration live in this notebook. Only runtime checks are imported from `src.validation.gold`. Execute weather → air quality → earthquakes. All persistent writes are restricted to the explicitly approved catalog's gold schema — `weather_api.gold` (production) or `weather_api_test.gold` (test). No job or dashboard is created.

## Parameters
`project_root` must contain this checkout and the notebook JSON sources. The catalog/schema parameters are validated against the explicitly approved target (`weather_api`/`gold` production, `weather_api_test`/`gold` test); a blank catalog fails validation instead of silently defaulting.

In [0]:
# Minimal source locator; all validation/loading rules live in Weather bootstrap.
from pathlib import Path
import json
import sys
dbutils.widgets.text('project_root','')
_override=dbutils.widgets.get('project_root')
if _override and not Path(_override).expanduser().is_absolute():raise ValueError('project_root must be absolute, or blank for discovery')
_candidates=[Path(_override).expanduser().resolve()] if _override else [Path.cwd().resolve(),*Path.cwd().resolve().parents]
_root=next((p for p in _candidates if (p/'src/config.py').is_file() and (p/'notebooks/gold/weather_gold.ipynb').is_file()),None)
if _root is None:raise FileNotFoundError('Provide absolute project_root with src and notebook JSON sources; Git-folder source export may be required')
_boot=json.loads((_root/'notebooks/gold/weather_gold.ipynb').read_text())
_boot_cells=[c for c in _boot['cells'] if c.get('cell_type')=='code' and c.get('metadata',{}).get('tags')==['gold_bootstrap']]
if len(_boot_cells)!=1:raise ValueError('Expected exactly one canonical gold_bootstrap cell')
exec(compile(''.join(_boot_cells[0]['source']),'weather_gold bootstrap','exec'),globals())
project_root=resolve_project_root(str(_root))
sys.path.insert(0,str(project_root))
dbutils.widgets.text('config_path','config/isolated.example.json')
dbutils.widgets.dropdown('validate_only','true',['true','false'])
config_path=resolve_config_path(project_root,dbutils.widgets.get('config_path'))
dbutils.widgets.text("release_id", "")

dbutils.widgets.text('gold_schema','gold')
dbutils.widgets.text('catalog','')  # required explicitly; approved: weather_api (production) or weather_api_test (test)


## Shared definitions
Loading definition cells has no database side effects. Successful persisted predecessors are checked again by this notebook.

In [0]:
load_gold_definitions(project_root,['weather', 'air_quality'],globals())


## Source selection and business transformations
All joins use full UTC timestamps and release identity. Source-specific expressions are owned here.

In [0]:
# gold_definitions: EARTHQUAKE-owned transformations, spatial products and final publication
SEISMIC_KEYS={'seismic_events_latest':['event_id'],'earthquake_snapshot_audit':['event_id','snapshot_at'],
              'seismic_coverage':['source','identity_key']}
SEISMIC_VIEW_KEYS={'seismic_daily_summary':['event_date_utc','event_type','magnitude_type'],
 'seismic_grid_daily':['event_date_utc','event_type','magnitude_type','grid_cell_id'],
 'location_seismic_nearby':['location_id','event_id'],'seismic_freshness':['source','identity_key'],
 'freshness_coverage':['source','identity_key']}

def seismic(relation, countries=None):
    """Latest event per event_id. Country attribution joins precomputed H3 resolution-6
    land cells derived from the documented public-domain Natural Earth 110m admin-0
    boundaries (reference.country_h3_cells). Events with no land cell (offshore or
    outside boundary coverage) carry explicit UNASSIGNED values; no NULL attribution."""
    latest_sql=latest(relation,['event_id'],'updated_at DESC,snapshot_at DESC,source_file_path DESC')
    if countries:
        attribution="""coalesce(c.country_code,'UNASSIGNED') country_code,
      coalesce(c.country_name,'Unassigned (offshore or outside land boundaries)') country_name"""
        joined=f' LEFT JOIN {countries} c ON c.cell=h3_longlatash3(s.longitude,s.latitude,6)'
    else:
        attribution="""'UNASSIGNED' country_code,
      'Unassigned (offshore or outside land boundaries)' country_name"""
        joined=''
    return f'''SELECT s.*,cast(event_time AS DATE) event_date_utc,
      CASE WHEN magnitude IS NULL THEN 'Unknown' WHEN magnitude<0 THEN '<0' WHEN magnitude<2 THEN '0-<2'
        WHEN magnitude<4 THEN '2-<4' WHEN magnitude<5 THEN '4-<5' WHEN magnitude<6 THEN '5-<6' WHEN magnitude<7 THEN '6-<7' ELSE '7+' END magnitude_band,
      CASE WHEN depth_km IS NULL THEN 'Unknown' WHEN depth_km<0 THEN 'negative_source_depth'
        WHEN depth_km<70 THEN '0-<70 km' WHEN depth_km<300 THEN '70-<300 km' ELSE '300+ km' END depth_band,
      {attribution}
      FROM ({latest_sql}) s{joined}'''

def audit(relation,columns):
    excluded={'source_file_path','_rescued_data','api_endpoint','source','feed_api_version','feed_title','ingested_at','snapshot_at','feed_generated_at','feed_generated_ms','updated_at','updated_at_ms'}
    attrs=[n for n in columns if n not in excluded]
    physical=[n for n in ('magnitude','magnitude_type','latitude','longitude','depth_km') if n in columns]
    def hashed(names):return 'sha2(to_json(named_struct('+','.join(literal(n)+','+('sort_array(array_distinct(`'+n+'`))' if n in {'all_event_ids','all_sources','available_product_types'} else '`'+n+'`') for n in names)+"),map('ignoreNullFields','false')),256)"
    return f'''WITH hashes AS (SELECT *,{hashed(attrs)} attribute_hash,{hashed(physical)} measurement_hash FROM {relation}),
      previous AS (SELECT *,row_number() OVER(PARTITION BY event_id ORDER BY snapshot_at) snapshot_number,
        lag(snapshot_at) OVER(PARTITION BY event_id ORDER BY snapshot_at) previous_snapshot_at,
        lag(updated_at_ms) OVER(PARTITION BY event_id ORDER BY snapshot_at) previous_updated_at_ms,
        lag(attribute_hash) OVER(PARTITION BY event_id ORDER BY snapshot_at) previous_attribute_hash,
        lag(measurement_hash) OVER(PARTITION BY event_id ORDER BY snapshot_at) previous_measurement_hash FROM hashes)
      SELECT *,snapshot_number=1 is_first_snapshot,
        CASE WHEN snapshot_number=1 THEN cast(NULL AS BOOLEAN) ELSE updated_at_ms<>previous_updated_at_ms END source_version_changed,
        CASE WHEN snapshot_number=1 THEN cast(NULL AS BOOLEAN) ELSE attribute_hash<>previous_attribute_hash END substantive_attributes_changed,
        CASE WHEN snapshot_number=1 THEN cast(NULL AS BOOLEAN) ELSE measurement_hash<>previous_measurement_hash END measurement_or_geometry_changed,
        CASE WHEN snapshot_number=1 THEN cast(NULL AS BOOLEAN) ELSE attribute_hash=previous_attribute_hash AND updated_at_ms=previous_updated_at_ms END unchanged_repeat FROM previous'''

def distance_sql(lat1,lon1,lat2,lon2):
    return f'''({2*EARTH_RADIUS_KM}*asin(sqrt(least(1D,greatest(0D,pow(sin(radians({lat2}-{lat1})/2),2)
      +cos(radians({lat1}))*cos(radians({lat2}))*pow(sin(radians({lon2}-{lon1})/2),2))))))'''

def seismic_coverage(ctx):
    feeds=[e for e in ctx['evidence'] if e['entry']['source']=='earthquakes']
    newest=max(feeds,key=lambda e:datetime.fromisoformat(e['snapshot_at'].replace('Z','+00:00'))) if feeds else None
    retrieved='cast('+literal(newest['snapshot_at'])+' AS TIMESTAMP)' if newest else 'cast(NULL AS TIMESTAMP)'
    generated='timestamp_millis('+str(newest['feed_generated_ms'])+')' if newest else 'cast(NULL AS TIMESTAMP)'
    return f'''SELECT cast(NULL AS BIGINT) location_id,'global' identity_key,'earthquakes' source,
      cast(NULL AS STRING) location_name,cast(NULL AS DOUBLE) latitude,cast(NULL AS DOUBLE) longitude,true enabled,
      {retrieved} current_retrieval_at,max(event_time) current_valid_at,{retrieved} latest_retrieval_at,max(event_time) latest_valid_at,
      min(event_time) coverage_start_at,max(event_time) coverage_end_at,count(*) retained_rows,
      count(magnitude) core_measurement_rows,count_if(magnitude IS NULL) missing_core_rows,
      cast(NULL AS BOOLEAN) current_missing_core,0 represented_locations,{generated} feed_generated_at
      FROM {ctx['stages']['seismic_events_latest']}'''

def earthquake_views(mapping):
    e=mapping['seismic_events_latest'];d=mapping['dim_location'];r=mapping['_gold_settings']
    measures='''count(DISTINCT event_id) event_count,count(magnitude) known_magnitude_count,max(magnitude) max_magnitude,
      count_if(magnitude<2) magnitude_below_2_count,count_if(magnitude>=2 AND magnitude<4) magnitude_2_to_4_count,
      count_if(magnitude>=4 AND magnitude<6) magnitude_4_to_6_count,count_if(magnitude>=6) magnitude_6_plus_count,
      count(depth_km) known_depth_count,min(depth_km) min_depth_km,max(depth_km) max_depth_km,cast(avg(cast(depth_km AS DECIMAL(30,12))) AS DOUBLE) mean_depth_km,
      count_if(review_status='reviewed') reviewed_count,count_if(review_status='automatic') automatic_count,
      count(felt_report_count) known_felt_report_count,sum(felt_report_count) total_reported_felt_count'''
    daily=f'''SELECT _release_id,event_date_utc,event_type,coalesce(magnitude_type,'Unknown') magnitude_type,{measures}
      FROM {e} GROUP BY _release_id,event_date_utc,event_type,coalesce(magnitude_type,'Unknown')'''
    size="cast(get_json_object(r.settings,'$.grid_degrees') AS DOUBLE)"
    grid=f'''WITH cells AS (SELECT e.*,{size} grid_degrees,
      cast(least(180/{size}-1,floor((latitude+90)/{size})) AS BIGINT) latitude_cell,
      cast(floor(pmod(longitude+180,360)/{size}) AS BIGINT) longitude_cell
      FROM {e} e JOIN {r} r ON e._release_id=r.release_id)
      SELECT _release_id,event_date_utc,event_type,coalesce(magnitude_type,'Unknown') magnitude_type,grid_degrees,latitude_cell,longitude_cell,
      concat(cast(grid_degrees AS STRING),':',latitude_cell,':',longitude_cell) grid_cell_id,
      -90+latitude_cell*grid_degrees latitude_min,-90+(latitude_cell+1)*grid_degrees latitude_max,
      -180+longitude_cell*grid_degrees longitude_min,-180+(longitude_cell+1)*grid_degrees longitude_max,
      -90+(latitude_cell+0.5)*grid_degrees latitude_center,-180+(longitude_cell+0.5)*grid_degrees longitude_center,{measures}
      FROM cells GROUP BY _release_id,event_date_utc,event_type,coalesce(magnitude_type,'Unknown'),grid_degrees,latitude_cell,longitude_cell'''
    radius="cast(get_json_object(r.settings,'$.nearby_radius_km') AS DOUBLE)";angle=f'({radius}/{EARTH_RADIUS_KM})'
    bridge=f'''WITH candidates AS (SELECT d._release_id,d.location_id,e.event_id,
      {distance_sql('d.latitude','d.longitude','e.latitude','e.longitude')} distance_km,{radius} radius_km,
      e.event_time,e.event_date_utc,e.event_type,e.magnitude,e.magnitude_type,e.depth_km,e.review_status,e.event_page_url
      FROM {d} d JOIN {e} e ON d._release_id=e._release_id JOIN {r} r ON d._release_id=r.release_id
      WHERE abs(e.latitude-d.latitude)<=degrees({angle}) AND (abs(radians(d.latitude))+{angle}>=pi()/2
        OR abs(pmod(e.longitude-d.longitude+540,360)-180)<=degrees(asin(least(1D,sin({angle})/cos(radians(d.latitude)))))))
      SELECT *,row_number() OVER(PARTITION BY _release_id,location_id ORDER BY distance_km,event_id) distance_rank
      FROM candidates WHERE distance_km<=radius_km'''
    both=f"(SELECT * FROM {mapping['weather_aq_coverage']} UNION ALL SELECT * FROM {mapping['seismic_coverage']})"
    return {'seismic_daily_summary':daily,'seismic_grid_daily':grid,'location_seismic_nearby':bridge,
      'seismic_freshness':freshness_query(mapping['seismic_coverage'],r),
      'freshness_coverage':freshness_query(both,r)}

def publication_status_query(cfg):
    return f'''SELECT r.release_id,r.built_at,r.published_at,r.input_versions,r.eligible_paths,
      coalesce(s.validated_stages,0) validated_stages,coalesce(s.failed_stages,0) failed_stages,
      (r.release_id=p.release_id AND p.control_valid AND r.receipt_count=1) is_active_release,
      CASE WHEN r.release_id IS NULL OR r.release_id='' OR NOT p.control_valid OR r.receipt_count<>1 OR coalesce(s.invalid_receipts,false) THEN 'INVALID_CONTROL_STATE'
        WHEN r.publication_state='LEGACY_ACTIVE' AND r.release_id=p.release_id THEN 'LEGACY_ACTIVE_UNVERIFIED'
        WHEN r.health='UNHEALTHY' THEN CASE WHEN r.release_id=p.release_id THEN 'UNHEALTHY_ACTIVE' ELSE 'UNHEALTHY' END
        WHEN r.publication_state='WITHDRAWN' THEN 'WITHDRAWN'
        WHEN s.failed_stages>0 OR r.publication_state='FAILED' THEN 'FAILED'
        WHEN r.release_id=p.release_id AND r.publication_state='PUBLISHED' AND s.validated_stages=3 THEN 'ACTIVE_VALIDATED_RELEASE'
        WHEN r.publication_state='PUBLISHED' AND s.validated_stages=3 THEN 'HISTORICALLY_PUBLISHED'
        WHEN r.release_id=p.release_id THEN 'ACTIVE_PENDING_VERIFICATION'
        WHEN s.validated_stages=3 THEN 'VALIDATED_NOT_ACTIVE' ELSE 'STAGED_INCOMPLETE' END publication_status
      FROM (SELECT *,count(*) OVER(PARTITION BY release_id) receipt_count,row_number() OVER(PARTITION BY release_id ORDER BY built_at DESC) receipt_number FROM {cfg.table('_gold_releases')}) r
      CROSS JOIN (SELECT max(release_id) release_id,count(*)=1 AND count_if(singleton=1)=1 AND count_if(release_id IS NULL)=0 control_valid FROM {cfg.table('_gold_active')}) p
      LEFT JOIN (SELECT release_id,count_if(status='VALIDATED') validated_stages,count_if(status='FAILED') failed_stages,
        (count(*)<>count(DISTINCT stage_name) OR count_if(stage_name IS NULL OR stage_name NOT IN ('weather','air_quality','earthquakes') OR status IS NULL OR status NOT IN ('BUILDING','VALIDATED','FAILED'))>0) invalid_receipts
        FROM {cfg.table('_gold_stages')} GROUP BY release_id) s ON r.release_id=s.release_id WHERE r.receipt_number=1'''

def verify_dependency_queries(ctx,queries):
    for name,q in queries.items():
        expected=ctx.temp('expected_dependency_'+name,f'SELECT {literal(ctx["release_id"])} _release_id,cast({literal(ctx["built_at"])} AS TIMESTAMP) _built_at,s.* FROM ({q}) s')
        checks.reconcile(ctx['db'],expected,ctx['stages'][name])



## Validation, publication and persistence reconciliation
All planned stage outputs are checked before stage copies are written. Failures propagate and leave the preceding active release visible.

In [0]:
def run_earthquakes(session,*,config_path,release_id,project_root='.',validate_only=True,gold=None):
    ctx=prepare_context(session,config_path,gold,release_id=release_id,project_root=project_root,write_stage=None if validate_only else 'earthquakes')
    try:
        checks.validate_stage_dependency(completed_stages(ctx),'earthquakes')
        persisted_dependencies(ctx,list(WEATHER_KEYS)+list(AQ_KEYS))
        # Completed Gold stage receipts alone are not proof of uncorrupted copies.
        verify_dependency_queries(ctx,weather_transformations(ctx))
        verify_dependency_queries(ctx,{'air_quality_'+kind+'_latest':air_quality_query(ctx['sources']['air_quality',kind],ctx['registry'],kind) for kind in ('current','hourly')})
        verify_dependency_queries(ctx,{'weather_aq_coverage':weather_aq_coverage(ctx)})
        history=ctx['sources']['earthquakes','history']
        counts=stage_products(ctx,{'seismic_events_latest':seismic(history,checks.country_cells_relation(ctx)),
          'earthquake_snapshot_audit':audit(history,list(output_types('earthquakes','history')))},SEISMIC_KEYS)
        # Country attribution is checked before any write: explicit UNASSIGNED, no NULLs,
        # assigned codes must be ISO 3166-1 alpha-3.
        # Flat int keys: Gold v2 controls validation requires every stage receipt count to be a non-negative int.
        counts.update({'country_attribution_'+k:v for k,v in checks.validate_country_attribution(ctx['db'],ctx['stages']['seismic_events_latest']).items()})
        checks.validate_latest(ctx['db'],history,ctx['stages']['seismic_events_latest'],['event_id'],seismic=True)
        counts.update(stage_products(ctx,{'seismic_coverage':seismic_coverage(ctx)},SEISMIC_KEYS))
        candidate=candidate_mapping(ctx)
        all_queries=air_quality_views(candidate)|earthquake_views(candidate)
        all_keys=AQ_VIEW_KEYS|SEISMIC_VIEW_KEYS
        derived,extra=build_derived(ctx,all_queries,all_keys);counts.update(extra)
        if not validate_only:
            planned=earthquake_views(public_mapping(ctx))|{n+'_history':q for n,q in earthquake_views(history_mapping(ctx)).items()}|{'published_releases':published_releases_query(ctx['cfg']),'publication_status':publication_status_query(ctx['cfg'])}
            preflight_query_family(ctx,planned)
            already_active=_active_id(ctx['db'],ctx['cfg'])==release_id
            if not already_active:stage_status(ctx,'earthquakes','BUILDING')
            try:
                preflight_and_write(ctx,SEISMIC_KEYS,SEISMIC_KEYS)
                install_views(ctx,earthquake_views(public_mapping(ctx)),SEISMIC_VIEW_KEYS)
                install_history_views(ctx,earthquake_views(history_mapping(ctx)),SEISMIC_VIEW_KEYS)
                install_views(ctx,{'published_releases':published_releases_query(ctx['cfg'])},{'published_releases':['release_id']})
                install_views(ctx,{'publication_status':publication_status_query(ctx['cfg'])},{'publication_status':['release_id']})
                # Rebind the same business SQL to persisted release copies and
                # reconcile EVERY derived product before activation, not afterward.
                copied_queries=air_quality_views(candidate_mapping(ctx))|earthquake_views(candidate_mapping(ctx))
                for name,q in copied_queries.items():
                    copied=ctx.temp('copy_bound_'+name,q)
                    checks.reconcile_derived(ctx['db'],derived[name],copied,freshness=('freshness' in name))
                checks.require_stable_versions(ctx['db'],ctx['versions'],_version)
                if not already_active:stage_status(ctx,'earthquakes','VALIDATED',counts)
                checks.validate_publication_completion(completed_stages(ctx))
                controls(ctx)
                pointer=require_writer(ctx);receipt=_record(ctx['db'],ctx['cfg'],release_id)
                checks.validate_candidate_fence(pointer,receipt)
                previous=receipt['expected_predecessor']
                def verify_active():
                    for name in ctx['stages']:checks.reconcile(ctx['db'],ctx['stages'][name],ctx['cfg'].table(name))
                    checks.validate_derived(ctx['db'],{n:ctx['cfg'].table(n) for n in ctx['stages']},
                        {n:ctx['cfg'].table(n) for n in all_queries})
                    for name in derived:
                        checks.reconcile_derived(ctx['db'],derived[name],ctx['cfg'].table(name),freshness=('freshness' in name or name=='freshness_coverage'))
                    checks.check_relation(ctx['db'],ctx['cfg'].table('publication_status'),['release_id'])
                if not already_active:
                    ctx['db'].query(f"UPDATE {ctx['cfg'].table('_gold_releases')} SET publication_state='READY',health='HEALTHY' WHERE release_id={literal(release_id)}")
                    activate_release(ctx['db'],ctx['cfg'],release_id,previous,verify_active,generation=receipt['expected_generation'],writer_token=ctx['writer_token'],verify_prior=lambda:verify_release_copies(ctx,previous))
                    ctx['db'].query(f"UPDATE {ctx['cfg'].table('_gold_releases')} SET publication_state='PUBLISHED',published_at=current_timestamp(),verified_at=current_timestamp() WHERE release_id={literal(release_id)}")
                else:
                    verify_active()
                    if receipt['publication_state']=='READY':
                        ctx['db'].query(f"UPDATE {ctx['cfg'].table('_gold_releases')} SET publication_state='PUBLISHED',published_at=current_timestamp(),verified_at=current_timestamp() WHERE release_id={literal(release_id)}")
            except Exception as exc:
                if not already_active and _active_id(ctx['db'],ctx['cfg'])!=release_id:stage_status(ctx,'earthquakes','FAILED',error=type(exc).__name__)
                raise
        return finish_result(ctx,{'stage':'earthquakes','release_id':release_id,'validate_only':validate_only,'published':not validate_only,'counts':counts,**ctx['report']})
    except Exception as exc:
        record_failure(ctx,exc);raise
    finally:ctx.close()


## Execute
Set `validate_only=false` deliberately to persist this stage. Pass the weather stage release_id unchanged to both later notebooks.

In [0]:
gold_result = run_earthquakes(
    spark, gold=GoldConfig(catalog=dbutils.widgets.get('catalog'), schema=dbutils.widgets.get('gold_schema')), config_path=config_path, project_root=project_root,
    release_id=dbutils.widgets.get("release_id"),
    validate_only=dbutils.widgets.get("validate_only") == "true",
)


## Summary
A validated stage is not an active complete release until all three notebooks finish. Inspect `publication_status` after final publication.

In [0]:
print(json.dumps(gold_result, default=str, sort_keys=True))
